In [ ]:
import time
import math
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Tuple
import sys, os

import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T_transforms
import soundfile as sf

log = logging.getLogger(__name__)

SAMPLE_RATE    = 16_000
FRAME_STRIDE   = 320          # CNN total stride — 20ms per frame at 16kHz
FRAME_RATE     = SAMPLE_RATE / FRAME_STRIDE   # 50 Hz


# ── Output types ──────────────────────────────────────────────────────────────

@dataclass
class WordResult:
    word:       str
    start_s:    float
    end_s:      float
    confidence: float   # mean frame probability of the emitted tokens, 0–1

    def __repr__(self):
        return f"WordResult({self.word!r}, {self.start_s:.2f}s–{self.end_s:.2f}s, conf={self.confidence:.2f})"


@dataclass
class TranscriptResult:
    text:          str
    words:         List[WordResult]
    duration_s:    float
    rtf:           float    # real-time factor: inference_time / audio_duration
    num_chunks:    int      # 1 for short audio, >1 for sliding window

    def __repr__(self):
        return (
            f"TranscriptResult(\n"
            f"  text={self.text!r}\n"
            f"  words={len(self.words)} words\n"
            f"  duration={self.duration_s:.2f}s  rtf={self.rtf:.3f}  chunks={self.num_chunks}\n"
            f")"
        )


@dataclass
class BenchmarkResult:
    mean_rtf:     float
    std_rtf:      float
    min_rtf:      float
    max_rtf:      float
    mean_ms:      float
    audio_duration_s: float
    runs:         int

    def __repr__(self):
        return (
            f"BenchmarkResult over {self.runs} runs on {self.audio_duration_s:.2f}s audio:\n"
            f"  RTF  mean={self.mean_rtf:.4f}  std={self.std_rtf:.4f}  "
            f"min={self.min_rtf:.4f}  max={self.max_rtf:.4f}\n"
            f"  Latency  mean={self.mean_ms:.1f}ms"
        )


# ── CTC greedy decoder with timestamps ───────────────────────────────────────

def ctc_decode_with_timestamps(
    log_probs:  torch.Tensor,    # (T, vocab_size)  — float32, CPU
    id2char:    dict,
    blank_id:   int,
    space_id:   int,
    frame_rate: float = FRAME_RATE,
    offset_s:   float = 0.0,
) -> Tuple[str, List[WordResult]]:

    T, V = log_probs.shape
    probs = log_probs.exp()    # (T, V)

    # ── Greedy collapse ───────────────────────────────────────────────────────
    # Build list of (token_id, first_frame, last_frame, mean_prob)
    tokens_with_frames = []
    prev_id = -1
    span_start = 0

    for t in range(T):
        tok = int(log_probs[t].argmax().item())
        prob = float(probs[t, tok].item())

        if tok == blank_id:
            prev_id = -1
            continue

        if tok != prev_id:
            tokens_with_frames.append([tok, t, t, prob, 1])
            prev_id = tok
        else:
            # Extend current span
            tokens_with_frames[-1][2] = t
            tokens_with_frames[-1][3] += prob
            tokens_with_frames[-1][4] += 1

    # Finalise mean confidence per token
    decoded_tokens = []
    for tok_id, f_start, f_end, prob_sum, count in tokens_with_frames:
        if tok_id == blank_id:
            continue
        decoded_tokens.append((tok_id, f_start, f_end, prob_sum / count))

    if not decoded_tokens:
        return "", []

    # ── Group into words by space token ──────────────────────────────────────
    words: List[WordResult] = []
    current_chars  = []
    current_frames = []
    current_confs  = []

    def flush_word():
        if not current_chars:
            return
        text = "".join(
            " " if id2char.get(c, "") == "<space>" else id2char.get(c, "?")
            for c in current_chars
        ).strip()
        if not text:
            return
        start_s = offset_s + current_frames[0]  / frame_rate
        end_s   = offset_s + current_frames[-1] / frame_rate
        conf    = float(np.mean(current_confs))
        words.append(WordResult(text, start_s, end_s, conf))
        current_chars.clear()
        current_frames.clear()
        current_confs.clear()

    for tok_id, f_start, f_end, conf in decoded_tokens:
        if tok_id == space_id:
            flush_word()
        else:
            current_chars.append(tok_id)
            current_frames.extend([f_start, f_end])
            current_confs.append(conf)

    flush_word()

    text = " ".join(w.word for w in words)
    return text, words


# ── Model loader ──────────────────────────────────────────────────────────────

def _load_model_and_tokenizer(checkpoint_path: str, tokenizer_path: str):
    """Load ASR model and tokenizer from disk."""
    from src.ssl_model import NepaliSSL, NepaliSSLConfig
    from src.asr_model import NepaliASR
    from src.tokenizer import GraphemeTokenizer

    tokenizer = GraphemeTokenizer.load(tokenizer_path)

    ckpt = torch.load(checkpoint_path, map_location="cpu")

    cfg_data = ckpt.get("config", None)
    if isinstance(cfg_data, dict):
        cfg = NepaliSSLConfig(**cfg_data)
    else:
        cfg = NepaliSSLConfig()

    ssl_model = NepaliSSL(cfg)

    # Strip ASR head weights to get just the encoder
    model_state = ckpt["model"]
    ssl_keys = {k: v for k, v in model_state.items()
                if k.startswith("encoder.") or k.startswith("context.")}

    # Build a temporary SSL model to hold encoder weights
    ssl_model.load_state_dict(
        {k.replace("encoder.", "cnn.").replace("context.", "context."): v
         for k, v in ssl_keys.items()},
        strict=False
    )

    asr_model = NepaliASR(ssl_model, vocab_size=tokenizer.get_vocab_size(), freeze_encoder=False)
    asr_model.load_state_dict(model_state)
    asr_model.eval()

    return asr_model, tokenizer


# ── Audio loading and preprocessing ──────────────────────────────────────────

def load_audio(path: str) -> Tuple[torch.Tensor, float]:
    data, sr = sf.read(path, dtype="float32", always_2d=True)
    # data: (T_samples, channels)

    # Convert stereo/multi-channel to mono
    if data.shape[1] > 1:
        data = data.mean(axis=1)
    else:
        data = data[:, 0]

    waveform = torch.from_numpy(data)   # (T_samples,)

    # Resample if needed
    if sr != SAMPLE_RATE:
        resampler = T_transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
        waveform  = resampler(waveform.unsqueeze(0)).squeeze(0)

    duration_s = waveform.shape[0] / SAMPLE_RATE
    return waveform, duration_s

def preprocess_for_inference(path: str, target_sr: int = 16000,
                              leading_silence_s: float = 0.3) -> torch.Tensor:
    data, sr = sf.read(path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(data.mean(axis=1))

    # Resample if needed
    if sr != target_sr:
        waveform = T_transforms.Resample(sr, target_sr)(waveform.unsqueeze(0)).squeeze(0)

    # Prepend silence to fix first-character skip
    silence = torch.zeros(int(leading_silence_s * target_sr))
    waveform = torch.cat([silence, waveform])

    # Normalize amplitude to match training distribution
    rms = waveform.pow(2).mean().sqrt()
    if rms > 1e-6:
        waveform = waveform * (0.1 / rms)   # target RMS ~0.1
        
    duration_s = waveform.shape[0] / SAMPLE_RATE

    return waveform, duration_s

class NepaliASRInference:

    def __init__(
        self,
        checkpoint:     str,
        tokenizer:      str,
        ssl_config      = None,
        max_duration_s: float = 20.0,
        window_s:       float = 18.0,
        overlap_s:      float = 2.0,
        num_threads:    Optional[int] = None,
    ):
        if num_threads is not None:
            torch.set_num_threads(num_threads)

        log.info(f"Loading checkpoint: {checkpoint}")
        self.model, self.tokenizer = _load_model_and_tokenizer(checkpoint, tokenizer)
        self.model.eval()

        # Optimise for CPU inference
        self.model = torch.jit.optimize_for_inference(torch.jit.script(self.model)) \
            if False else self.model

        self.max_duration_s = max_duration_s
        self.window_s       = window_s
        self.overlap_s      = overlap_s

        self.blank_id = self.tokenizer.blank_id
        self.space_id = self.tokenizer.char2id[self.tokenizer.SPACE_TOKEN]
        self.id2char  = self.tokenizer.id2char

        log.info(
            f"Model ready — vocab={self.tokenizer.get_vocab_size()} "
            f"blank={self.blank_id} space={self.space_id} "
            f"threads={torch.get_num_threads()}"
        )


    @torch.no_grad()
    def _infer_chunk(
        self,
        waveform: torch.Tensor,   # (T_samples,) float32
        offset_s: float = 0.0,
    ) -> Tuple[str, List[WordResult]]:
        x       = waveform.unsqueeze(0)   # (1, T)
        lengths = torch.tensor([waveform.shape[0]], dtype=torch.long)

        log_probs, frame_lengths = self.model(x, lengths)
        # log_probs: (1, T_frames, vocab)

        T_valid   = int(frame_lengths[0].item())
        lp_slice  = log_probs[0, :T_valid].float().cpu()   # (T_valid, vocab)

        text, words = ctc_decode_with_timestamps(
            lp_slice, self.id2char, self.blank_id, self.space_id,
            frame_rate=FRAME_RATE, offset_s=offset_s,
        )
        return text, words


    def _infer_sliding(
        self,
        waveform:   torch.Tensor,
        duration_s: float,
    ) -> Tuple[str, List[WordResult], int]:

        window_samples  = int(self.window_s  * SAMPLE_RATE)
        overlap_samples = int(self.overlap_s * SAMPLE_RATE)
        step_samples    = window_samples - overlap_samples

        all_words: List[WordResult] = []
        chunk_idx = 0
        pos = 0

        while pos < waveform.shape[0]:
            end = min(pos + window_samples, waveform.shape[0])
            chunk = waveform[pos:end]

            # Pad short final chunk to at least 1 frame
            if chunk.shape[0] < FRAME_STRIDE:
                break

            offset_s = pos / SAMPLE_RATE
            _, words = self._infer_chunk(chunk, offset_s=offset_s)

            if chunk_idx > 0 and all_words:
                prev_end_s  = (pos + overlap_samples) / SAMPLE_RATE
                cutoff_s    = prev_end_s - self.overlap_s / 2
                words = [w for w in words if w.start_s >= cutoff_s]

            all_words.extend(words)
            chunk_idx += 1

            if end >= waveform.shape[0]:
                break
            pos += step_samples

        full_text = " ".join(w.word for w in all_words)
        return full_text, all_words, chunk_idx

    def transcribe(self, audio_path: str) -> TranscriptResult:

        waveform, duration_s = preprocess_for_inference(audio_path)

        t_start = time.perf_counter()

        if duration_s <= self.max_duration_s:
            text, words = self._infer_chunk(waveform, offset_s=0.0)
            num_chunks  = 1
        else:
            log.info(
                f"Audio {duration_s:.1f}s > {self.max_duration_s}s — "
                f"using sliding window ({self.window_s}s / {self.overlap_s}s overlap)"
            )
            text, words, num_chunks = self._infer_sliding(waveform, duration_s)

        elapsed_s = time.perf_counter() - t_start
        rtf       = elapsed_s / max(duration_s, 1e-6)

        return TranscriptResult(
            text       = text,
            words      = words,
            duration_s = duration_s,
            rtf        = rtf,
            num_chunks = num_chunks,
        )

    def transcribe_waveform(
        self,
        waveform:   torch.Tensor,   # (T_samples,) float32, already 16kHz
        duration_s: Optional[float] = None,
    ) -> TranscriptResult:

        if duration_s is None:
            duration_s = waveform.shape[0] / SAMPLE_RATE

        t_start = time.perf_counter()

        if duration_s <= self.max_duration_s:
            text, words = self._infer_chunk(waveform, offset_s=0.0)
            num_chunks  = 1
        else:
            text, words, num_chunks = self._infer_sliding(waveform, duration_s)

        elapsed_s = time.perf_counter() - t_start
        rtf       = elapsed_s / max(duration_s, 1e-6)

        return TranscriptResult(
            text       = text,
            words      = words,
            duration_s = duration_s,
            rtf        = rtf,
            num_chunks = num_chunks,
        )

    def benchmark(self, audio_path: str, runs: int = 20, warmup: int = 3) -> BenchmarkResult:

        waveform, duration_s = preprocess_for_inference(audio_path)

        log.info(f"Benchmarking on {duration_s:.2f}s audio — {warmup} warmup + {runs} timed runs")

        # Warmup
        for _ in range(warmup):
            self.transcribe_waveform(waveform, duration_s)

        # Timed runs
        times = []
        for i in range(runs):
            t0 = time.perf_counter()
            self.transcribe_waveform(waveform, duration_s)
            times.append(time.perf_counter() - t0)

        times    = np.array(times)
        rtfs     = times / duration_s

        result = BenchmarkResult(
            mean_rtf         = float(rtfs.mean()),
            std_rtf          = float(rtfs.std()),
            min_rtf          = float(rtfs.min()),
            max_rtf          = float(rtfs.max()),
            mean_ms          = float(times.mean() * 1000),
            audio_duration_s = duration_s,
            runs             = runs,
        )
        print(result)
        return result

    def profile_components(self, audio_path: str) -> dict:

        waveform, duration_s = preprocess_for_inference(audio_path)
        x       = waveform.unsqueeze(0)
        lengths = torch.tensor([waveform.shape[0]], dtype=torch.long)

        timings = {}

        with torch.no_grad():
            # CNN encoder
            t0 = time.perf_counter()
            features = self.model.encoder(x)
            timings["cnn_ms"] = (time.perf_counter() - t0) * 1000

            T = features.shape[1]
            frame_lengths = (lengths.float() / x.shape[1] * T).long().clamp(max=T)
            pad_mask = (
                torch.arange(T).unsqueeze(0) >= frame_lengths.unsqueeze(1)
            )

            # Transformer
            t0 = time.perf_counter()
            context = self.model.context(features, padding_mask=pad_mask)
            timings["transformer_ms"] = (time.perf_counter() - t0) * 1000

            # CTC head + softmax
            t0 = time.perf_counter()
            context_normed = self.model.pre_head_norm(context)
            logits    = self.model.ctc_head(context_normed)
            log_probs = F.log_softmax(logits, dim=-1)
            timings["ctc_head_ms"] = (time.perf_counter() - t0) * 1000

            # Decode
            T_valid  = int(frame_lengths[0].item())
            lp_slice = log_probs[0, :T_valid].float().cpu()
            t0 = time.perf_counter()
            ctc_decode_with_timestamps(
                lp_slice, self.id2char, self.blank_id, self.space_id
            )
            timings["decode_ms"] = (time.perf_counter() - t0) * 1000

        timings["total_ms"]      = sum(timings.values())
        timings["audio_duration_s"] = duration_s
        timings["rtf"]           = timings["total_ms"] / 1000 / duration_s

        print(f"\nComponent breakdown ({duration_s:.2f}s audio):")
        for k, v in timings.items():
            if k.endswith("_ms"):
                pct = v / timings["total_ms"] * 100
                print(f"  {k:<20s} {v:7.2f} ms  ({pct:.1f}%)")
        print(f"  {'rtf':<20s} {timings['rtf']:.4f}")

        return timings


# ── CLI ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import json

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

    audio = os.path.join("benchmark", "audio_2.wav")
    checkpoint = os.path.join("deployment", "asr_model_prototype.pt")
    tokenizer = os.path.join("data", "tokenizer.json")
    max_duration = 20.0
    window = 18.0
    overlap = 2.0
    threads = None
    profile = False
    benchmark = 0
    store_json = False

    asr = NepaliASRInference(
        checkpoint     = checkpoint,
        tokenizer      = tokenizer,
        max_duration_s = max_duration,
        window_s       = window,
        overlap_s      = overlap,
        num_threads    = threads,
    )

    if profile:
        asr.profile_components(audio)

    elif benchmark > 0:
        asr.benchmark(audio, runs=benchmark)

    else:
        result = asr.transcribe(audio)

        if store_json:
            out = {
                "text":       result.text,
                "duration_s": result.duration_s,
                "rtf":        result.rtf,
                "num_chunks": result.num_chunks,
                "words": [
                    {
                        "word":       w.word,
                        "start_s":    round(w.start_s, 3),
                        "end_s":      round(w.end_s, 3),
                        "confidence": round(w.confidence, 4),
                    }
                    for w in result.words
                ],
            }
            print(json.dumps(out, ensure_ascii=False, indent=2))
        else:
            print(f"\nTranscript:\n{result.text}\n")
            print(f"Duration: {result.duration_s:.2f}s  |  RTF: {result.rtf:.4f}  |  Chunks: {result.num_chunks}")
            print(f"\nWord timestamps:")
            for w in result.words:
                print(f"  [{w.start_s:6.2f}s – {w.end_s:6.2f}s]  {w.word:<30s}  conf={w.confidence:.2f}")


In [2]:
import time
import math
import logging
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Tuple
import sys, os

import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T_transforms
import soundfile as sf

log = logging.getLogger(__name__)

SAMPLE_RATE    = 16_000
FRAME_STRIDE   = 320
FRAME_RATE     = SAMPLE_RATE / FRAME_STRIDE


@dataclass
class WordResult:
    word:       str
    start_s:    float
    end_s:      float
    confidence: float

    def __repr__(self):
        return f"WordResult({self.word!r}, {self.start_s:.2f}s-{self.end_s:.2f}s, conf={self.confidence:.2f})"


@dataclass
class TranscriptResult:
    text:          str
    words:         List[WordResult]
    duration_s:    float
    rtf:           float
    num_chunks:    int

    def __repr__(self):
        return (
            f"TranscriptResult(\n"
            f"  text={self.text!r}\n"
            f"  words={len(self.words)} words\n"
            f"  duration={self.duration_s:.2f}s  rtf={self.rtf:.3f}  chunks={self.num_chunks}\n"
            f")"
        )


@dataclass
class BenchmarkResult:
    mean_rtf:         float
    std_rtf:          float
    min_rtf:          float
    max_rtf:          float
    mean_ms:          float
    audio_duration_s: float
    runs:             int

    def __repr__(self):
        return (
            f"BenchmarkResult over {self.runs} runs on {self.audio_duration_s:.2f}s audio:\n"
            f"  RTF  mean={self.mean_rtf:.4f}  std={self.std_rtf:.4f}  "
            f"min={self.min_rtf:.4f}  max={self.max_rtf:.4f}\n"
            f"  Latency  mean={self.mean_ms:.1f}ms"
        )


def ctc_decode_with_timestamps(
    log_probs:  torch.Tensor,
    id2char:    dict,
    blank_id:   int,
    space_id:   int,
    frame_rate: float = FRAME_RATE,
    offset_s:   float = 0.0,
) -> tuple:

    T, V = log_probs.shape
    probs = log_probs.exp()

    tokens_with_frames = []
    prev_id = -1

    for t in range(T):
        tok  = int(log_probs[t].argmax().item())
        prob = float(probs[t, tok].item())

        if tok == blank_id:
            prev_id = -1
            continue

        if tok != prev_id:
            tokens_with_frames.append([tok, t, t, prob, 1])
            prev_id = tok
        else:
            tokens_with_frames[-1][2]  = t
            tokens_with_frames[-1][3] += prob
            tokens_with_frames[-1][4] += 1

    decoded_tokens = []
    for tok_id, f_start, f_end, prob_sum, count in tokens_with_frames:
        if tok_id == blank_id:
            continue
        decoded_tokens.append((tok_id, f_start, f_end, prob_sum / count))

    if not decoded_tokens:
        return "", []

    words          = []
    current_chars  = []
    current_frames = []
    current_confs  = []

    def flush_word():
        if not current_chars:
            return
        text = "".join(
            " " if id2char.get(c, "") == "<space>" else id2char.get(c, "?")
            for c in current_chars
        ).strip()
        if not text:
            return
        start_s = offset_s + current_frames[0]  / frame_rate
        end_s   = offset_s + current_frames[-1] / frame_rate
        conf    = float(np.mean(current_confs))
        words.append(WordResult(text, start_s, end_s, conf))
        current_chars.clear()
        current_frames.clear()
        current_confs.clear()

    for tok_id, f_start, f_end, conf in decoded_tokens:
        if tok_id == space_id:
            flush_word()
        else:
            current_chars.append(tok_id)
            current_frames.extend([f_start, f_end])
            current_confs.append(conf)

    flush_word()

    text = " ".join(w.word for w in words)
    return text, words


def _load_model_and_tokenizer(checkpoint_path: str, tokenizer_path: str):
    from src.ssl_model import NepaliSSL, NepaliSSLConfig
    from src.asr_model import NepaliASR
    from src.tokenizer import GraphemeTokenizer

    tokenizer = GraphemeTokenizer.load(tokenizer_path)
    ckpt      = torch.load(checkpoint_path, map_location="cpu")

    cfg_data = ckpt.get("config", None)
    cfg      = NepaliSSLConfig(**cfg_data) if isinstance(cfg_data, dict) else NepaliSSLConfig()

    ssl_model   = NepaliSSL(cfg)
    model_state = ckpt["model"]
    ssl_keys    = {k: v for k, v in model_state.items()
                   if k.startswith("encoder.") or k.startswith("context.")}

    ssl_model.load_state_dict(
        {k.replace("encoder.", "cnn.").replace("context.", "context."): v
         for k, v in ssl_keys.items()},
        strict=False,
    )

    asr_model = NepaliASR(ssl_model, vocab_size=tokenizer.get_vocab_size(), freeze_encoder=False)
    asr_model.load_state_dict(model_state)
    asr_model.eval()

    return asr_model, tokenizer


def load_audio(path: str) -> tuple:
    """Raw load — resample to 16kHz mono. No preprocessing applied."""
    data, sr = sf.read(path, dtype="float32", always_2d=True)
    data     = data.mean(axis=1) if data.shape[1] > 1 else data[:, 0]
    waveform = torch.from_numpy(data)

    if sr != SAMPLE_RATE:
        waveform = T_transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)(
            waveform.unsqueeze(0)
        ).squeeze(0)

    return waveform, waveform.shape[0] / SAMPLE_RATE


def preprocess_waveform(
    waveform:          torch.Tensor,
    leading_silence_s: float = 0.3,
    target_rms:        float = 0.1,
) -> tuple:
    """
    Apply inference-time preprocessing to a 16kHz mono waveform tensor.

    1. Prepend leading silence — compensates for CTC first-character skip
       caused by speech onset timing mismatch with training data.
    2. RMS-normalize amplitude — matches training distribution (OpenSLR-54
       studio recordings) which avoids CNN feature scale mismatch on
       louder or quieter recordings.

    Parameters
    ----------
    waveform          : (T_samples,) float32 tensor at 16kHz
    leading_silence_s : seconds of silence to prepend (default 0.3)
    target_rms        : target RMS amplitude (default 0.1)

    Returns (preprocessed_waveform, duration_s).
    Can be called directly when working with waveform tensors.
    """
    silence  = torch.zeros(int(leading_silence_s * SAMPLE_RATE))
    waveform = torch.cat([silence, waveform])

    rms = waveform.pow(2).mean().sqrt()
    if rms > 1e-6:
        waveform = waveform * (target_rms / rms)

    return waveform, waveform.shape[0] / SAMPLE_RATE


def preprocess_for_inference(
    path:              str,
    leading_silence_s: float = 0.3,
    target_rms:        float = 0.1,
) -> tuple:
    """Load audio file and apply preprocessing. Returns (waveform, duration_s)."""
    waveform, _ = load_audio(path)
    return preprocess_waveform(waveform, leading_silence_s, target_rms)


class NepaliASRInference:
    """
    Deployment-ready Nepali ASR inference — CPU optimised.

    Parameters
    ----------
    checkpoint        : path to checkpoint_XXXXXXX_best.pt
    tokenizer         : path to tokenizer.json
    max_duration_s    : max audio length before sliding window kicks in (default 20s)
    window_s          : sliding window length in seconds (default 18s)
    overlap_s         : overlap between adjacent windows in seconds (default 2s)
    leading_silence_s : silence prepended to every clip to fix CTC onset skip (default 0.3s)
    target_rms        : RMS amplitude target for normalisation (default 0.1)
    num_threads       : PyTorch CPU thread count (default: all available)
    """

    def __init__(
        self,
        checkpoint:        str,
        tokenizer:         str,
        ssl_config         = None,
        max_duration_s:    float = 4.0,
        window_s:          float = 3.5,
        overlap_s:         float = 1.5,
        leading_silence_s: float = 0.3,
        target_rms:        float = 0.03,
        num_threads:       Optional[int] = None,
    ):
        if num_threads is not None:
            torch.set_num_threads(num_threads)

        log.info(f"Loading checkpoint: {checkpoint}")
        self.model, self.tokenizer = _load_model_and_tokenizer(checkpoint, tokenizer)
        self.model.eval()

        self.max_duration_s    = max_duration_s
        self.window_s          = window_s
        self.overlap_s         = overlap_s
        self.leading_silence_s = leading_silence_s
        self.target_rms        = target_rms
        self.blank_id          = self.tokenizer.blank_id
        self.space_id          = self.tokenizer.char2id[self.tokenizer.SPACE_TOKEN]
        self.id2char           = self.tokenizer.id2char

        log.info(
            f"Model ready — vocab={self.tokenizer.get_vocab_size()} "
            f"blank={self.blank_id} space={self.space_id} "
            f"threads={torch.get_num_threads()}"
        )

    @torch.no_grad()
    def _infer_chunk(self, waveform: torch.Tensor, offset_s: float = 0.0) -> tuple:
        x       = waveform.unsqueeze(0)
        lengths = torch.tensor([waveform.shape[0]], dtype=torch.long)

        log_probs, frame_lengths = self.model(x, lengths)

        T_valid  = int(frame_lengths[0].item())
        lp_slice = log_probs[0, :T_valid].float().cpu()

        return ctc_decode_with_timestamps(
            lp_slice, self.id2char, self.blank_id, self.space_id,
            frame_rate=FRAME_RATE, offset_s=offset_s,
        )

    def _infer_sliding(self, waveform: torch.Tensor, duration_s: float) -> tuple:
        window_samples  = int(self.window_s  * SAMPLE_RATE)
        overlap_samples = int(self.overlap_s * SAMPLE_RATE)
        step_samples    = window_samples - overlap_samples

        all_words = []
        chunk_idx = 0
        pos       = 0

        while pos < waveform.shape[0]:
            end   = min(pos + window_samples, waveform.shape[0])
            chunk = waveform[pos:end]

            if chunk.shape[0] < FRAME_STRIDE:
                break

            offset_s = pos / SAMPLE_RATE
            _, words = self._infer_chunk(chunk, offset_s=offset_s)

            # trim overlap head from all chunks except first
            if chunk_idx > 0 and words:
                cutoff_s = offset_s + self.overlap_s / 2  # simpler and correct
                words = [w for w in words if w.start_s >= cutoff_s]

            # trim overlap tail from all chunks except last
            is_last = (end >= waveform.shape[0])
            if not is_last and words:
                tail_cutoff_s = (pos + window_samples - overlap_samples / 2) / SAMPLE_RATE
                words = [w for w in words if w.end_s <= tail_cutoff_s]

            all_words.extend(words)
            chunk_idx += 1

            if end >= waveform.shape[0]:
                break
            pos += step_samples

        return " ".join(w.word for w in all_words), all_words, chunk_idx

    def _run(self, waveform: torch.Tensor, duration_s: float) -> tuple:
        if duration_s <= self.max_duration_s:
            text, words = self._infer_chunk(waveform, offset_s=0.0)
            return text, words, 1
        log.info(
            f"Audio {duration_s:.1f}s > {self.max_duration_s}s — "
            f"sliding window ({self.window_s}s / {self.overlap_s}s overlap)"
        )
        return self._infer_sliding(waveform, duration_s)

    def transcribe(self, audio_path: str) -> TranscriptResult:
        """
        Transcribe an audio file.
        Preprocessing (silence prepend + RMS normalisation) is applied automatically.
        """
        waveform, duration_s = preprocess_for_inference(
            audio_path, self.leading_silence_s, self.target_rms
        )
        t0 = time.perf_counter()
        text, words, num_chunks = self._run(waveform, duration_s)
        elapsed_s = time.perf_counter() - t0

        return TranscriptResult(
            text=text, words=words, duration_s=duration_s,
            rtf=elapsed_s / max(duration_s, 1e-6), num_chunks=num_chunks,
        )

    def transcribe_waveform(
        self,
        waveform:   torch.Tensor,
        duration_s: Optional[float] = None,
        preprocess: bool = True,
    ) -> TranscriptResult:
        """
        Transcribe a pre-loaded 16kHz mono waveform tensor.

        Parameters
        ----------
        preprocess : apply silence prepend + RMS normalisation (default True).
                     Set False only if you have already called preprocess_waveform().
        """
        if preprocess:
            waveform, duration_s = preprocess_waveform(
                waveform, self.leading_silence_s, self.target_rms
            )
        elif duration_s is None:
            duration_s = waveform.shape[0] / SAMPLE_RATE

        t0 = time.perf_counter()
        text, words, num_chunks = self._run(waveform, duration_s)
        elapsed_s = time.perf_counter() - t0

        return TranscriptResult(
            text=text, words=words, duration_s=duration_s,
            rtf=elapsed_s / max(duration_s, 1e-6), num_chunks=num_chunks,
        )

    def benchmark(self, audio_path: str, runs: int = 20, warmup: int = 3) -> BenchmarkResult:
        waveform, duration_s = preprocess_for_inference(
            audio_path, self.leading_silence_s, self.target_rms
        )
        log.info(f"Benchmarking on {duration_s:.2f}s audio — {warmup} warmup + {runs} timed runs")

        for _ in range(warmup):
            self.transcribe_waveform(waveform, duration_s, preprocess=False)

        times = []
        for _ in range(runs):
            t0 = time.perf_counter()
            self.transcribe_waveform(waveform, duration_s, preprocess=False)
            times.append(time.perf_counter() - t0)

        times = np.array(times)
        rtfs  = times / duration_s

        result = BenchmarkResult(
            mean_rtf=float(rtfs.mean()), std_rtf=float(rtfs.std()),
            min_rtf=float(rtfs.min()),   max_rtf=float(rtfs.max()),
            mean_ms=float(times.mean() * 1000),
            audio_duration_s=duration_s, runs=runs,
        )
        print(result)
        return result

    def transcribe_batch(
        self,
        audio_dir:  str,
        extensions: tuple = (".wav", ".flac", ".mp3", ".ogg", ".m4a"),
        recursive:  bool  = False,
    ) -> List[dict]:
        """
        Transcribe all audio files in a directory.

        Parameters
        ----------
        audio_dir  : path to directory containing audio files
        extensions : file extensions to include (default: wav, flac, mp3, ogg, m4a)
        recursive  : if True, also scan subdirectories (default False)

        Returns
        -------
        List of dicts with keys: file, transcript, duration_s, rtf, num_chunks, error
        Results are also printed as they complete.
        """
        audio_dir = Path(audio_dir)
        if not audio_dir.is_dir():
            raise ValueError(f"Not a directory: {audio_dir}")

        pattern = "**/*" if recursive else "*"
        files   = sorted([
            p for p in audio_dir.glob(pattern)
            if p.suffix.lower() in extensions and p.is_file()
        ])

        if not files:
            log.warning(f"No audio files found in {audio_dir}")
            return []

        log.info(f"Found {len(files)} audio file(s) in {audio_dir}")

        results      = []
        total_audio  = 0.0
        total_infer  = 0.0
        n_ok         = 0
        n_err        = 0

        for i, fpath in enumerate(files, 1):
            try:
                t0     = time.perf_counter()
                result = self.transcribe(str(fpath))
                elapsed_s = time.perf_counter() - t0

                total_audio += result.duration_s
                total_infer += elapsed_s
                n_ok        += 1

                row = {
                    "file":       str(fpath),
                    "transcript": result.text,
                    "duration_s": round(result.duration_s, 2),
                    "rtf":        round(result.rtf, 4),
                    "num_chunks": result.num_chunks,
                    "error":      None,
                }
                print(
                    f"[{i:>4}/{len(files)}]  {fpath.name:<40s}  "
                    f"{result.duration_s:5.1f}s  RTF={result.rtf:.3f}  "
                    f"{result.text[:60]!r}"
                )

            except Exception as e:
                n_err += 1
                row = {
                    "file":       str(fpath),
                    "transcript": None,
                    "duration_s": None,
                    "rtf":        None,
                    "num_chunks": None,
                    "error":      str(e),
                }
                log.warning(f"[{i:>4}/{len(files)}]  {fpath.name}  ERROR: {e}")

            results.append(row)

        # Summary
        if n_ok > 0:
            overall_rtf = total_infer / max(total_audio, 1e-6)
            print(
                f"\nBatch complete — {n_ok} ok / {n_err} errors  |  "
                f"total audio: {total_audio:.1f}s  |  "
                f"total inference: {total_infer:.1f}s  |  "
                f"overall RTF: {overall_rtf:.4f}"
            )

        return results

    def benchmark_batch(
        self,
        audio_dir:  str,
        extensions: tuple = (".wav", ".flac", ".mp3", ".ogg", ".m4a"),
        recursive:  bool  = False,
        runs:       int   = 5,
        warmup:     int   = 1,
    ) -> dict:
        """
        Benchmark inference latency across all files in a directory.

        Runs each file `runs` times (after `warmup` untimed runs) and
        aggregates RTF statistics across the whole set.

        Parameters
        ----------
        audio_dir  : path to directory containing audio files
        extensions : file extensions to include
        recursive  : scan subdirectories (default False)
        runs       : timed runs per file (default 5)
        warmup     : untimed warmup runs per file (default 1)

        Returns
        -------
        dict with per-file BenchmarkResult objects and aggregate stats
        """
        audio_dir = Path(audio_dir)
        if not audio_dir.is_dir():
            raise ValueError(f"Not a directory: {audio_dir}")

        pattern = "**/*" if recursive else "*"
        files   = sorted([
            p for p in audio_dir.glob(pattern)
            if p.suffix.lower() in extensions and p.is_file()
        ])

        if not files:
            log.warning(f"No audio files found in {audio_dir}")
            return {}

        log.info(
            f"Benchmarking {len(files)} file(s) — "
            f"{warmup} warmup + {runs} timed runs each"
        )

        per_file  = {}
        all_rtfs  = []

        for i, fpath in enumerate(files, 1):
            print(f"\n[{i}/{len(files)}] {fpath.name}")
            try:
                result = self.benchmark(str(fpath), runs=runs, warmup=warmup)
                per_file[str(fpath)] = result
                all_rtfs.extend([result.mean_rtf] * runs)
            except Exception as e:
                log.warning(f"  ERROR: {e}")
                per_file[str(fpath)] = None

        if all_rtfs:
            arr = np.array(all_rtfs)
            print(
                f"\nAggregate across {len(files)} file(s) ({runs} runs each):\n"
                f"  RTF  mean={arr.mean():.4f}  std={arr.std():.4f}  "
                f"min={arr.min():.4f}  max={arr.max():.4f}"
            )

        return {
            "per_file":      per_file,
            "aggregate_rtf": {
                "mean": float(np.mean(all_rtfs)) if all_rtfs else None,
                "std":  float(np.std(all_rtfs))  if all_rtfs else None,
                "min":  float(np.min(all_rtfs))  if all_rtfs else None,
                "max":  float(np.max(all_rtfs))  if all_rtfs else None,
            },
        }

    def profile_components(self, audio_path: str) -> dict:
        waveform, duration_s = preprocess_for_inference(
            audio_path, self.leading_silence_s, self.target_rms
        )
        x       = waveform.unsqueeze(0)
        lengths = torch.tensor([waveform.shape[0]], dtype=torch.long)
        timings = {}

        with torch.no_grad():
            t0 = time.perf_counter()
            features = self.model.encoder(x)
            timings["cnn_ms"] = (time.perf_counter() - t0) * 1000

            T             = features.shape[1]
            frame_lengths = (lengths.float() / x.shape[1] * T).long().clamp(max=T)
            pad_mask      = torch.arange(T).unsqueeze(0) >= frame_lengths.unsqueeze(1)

            t0 = time.perf_counter()
            context = self.model.context(features, padding_mask=pad_mask)
            timings["transformer_ms"] = (time.perf_counter() - t0) * 1000

            t0             = time.perf_counter()
            logits         = self.model.ctc_head(self.model.pre_head_norm(context))
            log_probs      = F.log_softmax(logits, dim=-1)
            timings["ctc_head_ms"] = (time.perf_counter() - t0) * 1000

            T_valid  = int(frame_lengths[0].item())
            lp_slice = log_probs[0, :T_valid].float().cpu()
            t0       = time.perf_counter()
            ctc_decode_with_timestamps(lp_slice, self.id2char, self.blank_id, self.space_id)
            timings["decode_ms"] = (time.perf_counter() - t0) * 1000

        timings["total_ms"]         = sum(timings.values())
        timings["audio_duration_s"] = duration_s
        timings["rtf"]              = timings["total_ms"] / 1000 / duration_s

        print(f"\nComponent breakdown ({duration_s:.2f}s audio):")
        for k, v in timings.items():
            if k.endswith("_ms"):
                print(f"  {k:<20s} {v:7.2f} ms  ({v / timings['total_ms'] * 100:.1f}%)")
        print(f"  {'rtf':<20s} {timings['rtf']:.4f}")
        return timings


if __name__ == "__main__":
    import json

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

    audio      = os.path.join("benchmark", "audio_2.wav")
    audio_dir  = os.path.join("benchmark")          # set to a directory path to run batch mode
    checkpoint = os.path.join("deployment", "asr_model_prototype.pt")
    tokenizer  = os.path.join("data", "tokenizer.json")
    profile      = False
    benchmark    = 0            # single-file benchmark runs (0 = off)
    batch        = True        # transcribe all files in audio_dir
    batch_bench  = False        # benchmark all files in audio_dir
    batch_runs   = 5            # runs per file in batch benchmark
    recursive    = False        # scan subdirectories
    store_json   = False

    asr = NepaliASRInference(checkpoint=checkpoint, tokenizer=tokenizer)

    # ── Batch modes ───────────────────────────────────────────────────────────
    if batch and audio_dir:
        results = asr.transcribe_batch(audio_dir, recursive=recursive)
        if store_json:
            out_path = Path(audio_dir) / "transcripts.json"
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"\nSaved to {out_path}")

    elif batch_bench and audio_dir:
        asr.benchmark_batch(audio_dir, runs=batch_runs, recursive=recursive)

    # ── Single file modes ─────────────────────────────────────────────────────
    elif profile:
        asr.profile_components(audio)

    elif benchmark > 0:
        asr.benchmark(audio, runs=benchmark)

    else:
        result = asr.transcribe(audio)
        if store_json:
            out = {
                "text": result.text, "duration_s": result.duration_s,
                "rtf": result.rtf,   "num_chunks": result.num_chunks,
                "words": [
                    {"word": w.word, "start_s": round(w.start_s, 3),
                     "end_s": round(w.end_s, 3), "confidence": round(w.confidence, 4)}
                    for w in result.words
                ],
            }
            print(json.dumps(out, ensure_ascii=False, indent=2))
        else:
            print(f"\nTranscript:\n{result.text}\n")
            print(f"Duration: {result.duration_s:.2f}s  |  RTF: {result.rtf:.4f}  |  Chunks: {result.num_chunks}")
            print(f"\nWord timestamps:")
            for w in result.words:
                print(f"  [{w.start_s:6.2f}s - {w.end_s:6.2f}s]  {w.word:<30s}  conf={w.confidence:.2f}")


2026-03-21 18:03:56,597 INFO Loading checkpoint: deployment\asr_model_prototype.pt
2026-03-21 18:03:56,801 INFO Model ready — vocab=723 blank=0 space=3 threads=4


Tokenizer loaded ← data\tokenizer.json  (723 tokens)


2026-03-21 18:03:56,804 INFO Found 24 audio file(s) in benchmark
2026-03-21 18:03:56,806 INFO Audio 4.3s > 4.0s — sliding window (3.5s / 1.5s overlap)
2026-03-21 18:03:57,191 INFO Audio 7.9s > 4.0s — sliding window (3.5s / 1.5s overlap)


[   1/24]  0565f0e9.wav                                4.3s  RTF=0.059  'दिपाधाअनिकोक्षणमा सुध्रपचिलेपालको बलनाङ जिल्लामा भएको हो'
[   2/24]  25fe7828.wav                                3.2s  RTF=0.040  'योकास'


2026-03-21 18:03:57,948 INFO Audio 4.3s > 4.0s — sliding window (3.5s / 1.5s overlap)


...

[  21/24]  d8ad54e5.wav                                6.7s  RTF=0.068  'पोकर फेमा तान आसपासको यो मानसपति पाइन्छ'


2026-03-21 18:04:09,349 INFO Audio 4.3s > 4.0s — sliding window (3.5s / 1.5s overlap)


[  22/24]  f6accdec.wav                                7.6s  RTF=0.074  'कलाको दौ सङ्ग्र छ'


2026-03-21 18:04:09,716 INFO Audio 8.7s > 4.0s — sliding window (3.5s / 1.5s overlap)


[  23/24]  fd86f5ba.wav                                4.3s  RTF=0.083  'दिपाधाअनिकोक्षणमा सुध्रपचिलेपालको बलनाङ जिल्लामा भएको हो'
[  24/24]  ffbd98e3.wav                                8.7s  RTF=0.067  'रखे जुमासड योभाउँलई पीपलकोटको मुख्यज्या सकिन्छ'

Batch complete — 24 ok / 0 errors  |  total audio: 185.6s  |  total inference: 13.5s  |  overall RTF: 0.0727
